# GPU Optimization for ragged

**Purpose:** Maximize performance with GPU acceleration

**Topics Covered:**
- GPU device detection
- Performance benchmarking
- Batch size optimization
- Memory management
- Multi-modal processing with vision embeddings

**Prerequisites:**
- GPU hardware (NVIDIA CUDA or Apple Silicon MPS)
- PyTorch with GPU support
- Sample PDFs with visual content

---

## 1. GPU Detection and Capabilities

First, verify that ragged detects your GPU correctly.

In [ ]:
# List available compute devices
!ragged gpu list

In [ ]:
# Show detailed GPU information
!ragged gpu info

In [ ]:
# Get JSON format for programmatic access
!ragged gpu info --format json

## 2. Performance Baseline

Establish baseline performance with and without GPU acceleration.

In [ ]:
# Run GPU benchmark
!ragged gpu benchmark --iterations 10

In [ ]:
# Compare CPU vs GPU performance
import time

# Test with GPU (default)
print("Testing with GPU acceleration...")
gpu_start = time.time()
!ragged ingest pdf ../sample_documents/data_visualization.pdf --vision
gpu_time = time.time() - gpu_start
print(f"GPU time: {gpu_time:.2f} seconds")

In [ ]:
# Test with CPU (force)
import os
os.environ['RAGGED_VISION_DEVICE'] = 'cpu'

print("Testing with CPU only...")
cpu_start = time.time()
!ragged clear --force
!ragged ingest pdf ../sample_documents/data_visualization.pdf --vision --vision-batch-size 1
cpu_time = time.time() - cpu_start
print(f"CPU time: {cpu_time:.2f} seconds")

# Calculate speedup
speedup = cpu_time / gpu_time
print(f"\nGPU Speedup: {speedup:.1f}x faster than CPU")

# Reset to GPU
os.environ['RAGGED_VISION_DEVICE'] = 'auto'

## 3. Batch Size Optimization

Find the optimal batch size for your GPU memory.

In [ ]:
# Auto-detect optimal batch size
!ragged gpu optimize-batch-size

In [ ]:
# Test different batch sizes
batch_sizes = [1, 2, 4, 8]
results = {}

for batch_size in batch_sizes:
    print(f"\nTesting batch size: {batch_size}")
    !ragged clear --force
    
    start = time.time()
    !ragged ingest pdf ../sample_documents/*.pdf --vision --vision-batch-size {batch_size}
    elapsed = time.time() - start
    
    results[batch_size] = elapsed
    print(f"Time: {elapsed:.2f}s")

# Display results
print("\nBatch Size Performance:")
print("=" * 40)
for batch_size, elapsed in sorted(results.items()):
    print(f"Batch {batch_size}: {elapsed:.2f}s")

## 4. Real-Time GPU Monitoring

Monitor GPU usage during processing.

In [ ]:
# Start GPU monitoring in background
# Note: This will print stats periodically
import subprocess
import threading

def monitor_gpu():
    subprocess.run(['ragged', 'gpu', 'stats', '--watch', '--interval', '2'])

# Start monitoring thread
monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()

print("GPU monitoring started...")

In [ ]:
# Run workload while monitoring
!ragged ingest pdf ../sample_documents/neural_networks.pdf --vision

## 5. Memory Management

Understand and optimize GPU memory usage.

In [ ]:
# Check current GPU memory usage
!ragged gpu stats

In [ ]:
# Monitor memory during batch processing
from ragged.gpu.device_manager import DeviceManager

dm = DeviceManager()

# Get memory info before processing
if dm.has_gpu:
    info = dm.get_device_info()
    print(f"GPU: {info.name}")
    print(f"Total Memory: {info.total_memory / 1024**3:.2f} GB")
    print(f"Available Memory: {info.available_memory / 1024**3:.2f} GB")
    print(f"Used Memory: {(info.total_memory - info.available_memory) / 1024**3:.2f} GB")
else:
    print("No GPU detected - using CPU")

## 6. Optimizing Vision Embeddings

Tune vision embedding generation for best performance.

In [ ]:
# Test different DPI settings (quality vs speed trade-off)
dpi_settings = [100, 150, 200]

for dpi in dpi_settings:
    print(f"\nTesting DPI: {dpi}")
    os.environ['RAGGED_VISION_PDF_DPI'] = str(dpi)
    
    !ragged clear --force
    start = time.time()
    !ragged ingest pdf ../sample_documents/data_visualization.pdf --vision
    elapsed = time.time() - start
    
    print(f"Time: {elapsed:.2f}s")

# Reset to default
os.environ['RAGGED_VISION_PDF_DPI'] = '150'

## 7. Programmatic GPU Control

Use Python API for fine-grained control.

In [ ]:
from ragged.gpu.device_manager import DeviceManager, DeviceType
from ragged.gpu.batch_sizer import BatchSizer

# Initialize device manager
dm = DeviceManager()

# Get device information
print("Available Devices:")
for device in dm.available_devices:
    print(f"  - {device.device_type.value}: {device.name}")
    if device.device_type != DeviceType.CPU:
        print(f"    Memory: {device.total_memory / 1024**3:.2f} GB")
        print(f"    Compute Capability: {device.compute_capability}")

In [ ]:
# Use BatchSizer for automatic optimization
if dm.has_gpu:
    sizer = BatchSizer(dm)
    recommended_batch = sizer.recommend_batch_size(
        model_memory_gb=2.0,  # ColPali model size
        image_size_mb=1.0     # Average page size
    )
    print(f"Recommended batch size: {recommended_batch}")
else:
    print("CPU mode - batch size: 1")

## 8. Performance Best Practices

Summary of optimization techniques.

### GPU Configuration Guide

**For 4GB VRAM:**
```bash
export RAGGED_VISION_BATCH_SIZE=1
export RAGGED_VISION_PDF_DPI=100
```

**For 8GB VRAM:**
```bash
export RAGGED_VISION_BATCH_SIZE=4
export RAGGED_VISION_PDF_DPI=150
```

**For 16GB+ VRAM:**
```bash
export RAGGED_VISION_BATCH_SIZE=8
export RAGGED_VISION_PDF_DPI=200
```

### Performance Tips

1. **Use GPU acceleration** when available (5-10x faster)
2. **Optimize batch size** based on VRAM
3. **Lower DPI** for faster processing (100-150 recommended)
4. **Monitor memory** to avoid OOM errors
5. **Batch documents** for better GPU utilization

### Troubleshooting

**Out of Memory (OOM):**
- Reduce batch size: `--vision-batch-size 1`
- Lower DPI: `export RAGGED_VISION_PDF_DPI=100`
- Close other GPU applications

**Slow Performance:**
- Verify GPU is being used: `ragged gpu stats`
- Increase batch size if memory allows
- Check PyTorch GPU installation

**GPU Not Detected:**
- Verify CUDA/MPS installation
- Check PyTorch version compatibility
- Update GPU drivers

## 9. Benchmark Results

Document your performance results for future reference.

In [ ]:
# Create performance report
import json
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "gpu_info": dm.get_device_info().__dict__ if dm.has_gpu else None,
    "batch_size_results": results,
    "recommended_config": {
        "batch_size": recommended_batch if dm.has_gpu else 1,
        "dpi": 150,
        "device": dm.primary_device.device_type.value if dm.has_gpu else "cpu"
    }
}

# Save report
with open('gpu_performance_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print("Performance report saved to gpu_performance_report.json")

## 10. Next Steps

**Advanced topics:**
- Multi-GPU support (future)
- Distributed processing
- Custom model quantization
- Production deployment optimization

**Related documentation:**
- [GPU Configuration Guide](../../docs/guides/gpu-configuration-optimisation.md)
- [Performance Tuning](../../docs/guides/performance-tuning.md)
- [Troubleshooting](../../docs/guides/troubleshooting.md)

---